In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
import pickle

import pandas as pd
import category_encoders as ce
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OneHotEncoder
from common import correlation_based_feature_selection as cbfs

In [ ]:
# Read in data
molecular_features = pd.read_pickle("molecular_features.pkl")
print(molecular_features.shape)

# 1. Input Features

## 1.1 Molecular Features

In [ ]:
## 1.1 Molecular Features  — MULTI-HOT (multi-label) encoding
X_molecular = molecular_features.copy()
X_molecular = X_molecular.drop(columns=["Progression Free Survival", "Event"])

discovery_molecular = X_molecular[X_molecular["Cohort"] == "Discovery"].copy()
replicate_molecular = X_molecular[X_molecular["Cohort"] == "Replicate"].copy()

# ── Config: raw token -> canonical alteration column ─────────────────────
# Edit here to split NF1 or break out MAPK variants.
TOKEN_MAP = {
    "KIAA1549-BRAF": "KIAA1549_BRAF",
    "BRAF V600E":    "BRAF_V600E",
    "NF1-germline":  "NF1",          # -> "NF1_germline" to split
    "NF1-somatic":   "NF1",          # -> "NF1_somatic"  to split
    "FGFR":          "FGFR",
    "RTK":           "RTK",
    "IDH":           "IDH",
    "MYB/MYBL1":     "MYB",
    "other MAPK":    "other_MAPK",
    "BRAF/MAPK":     "other_MAPK",
    "MAPK":          "other_MAPK",
    "CDKN2A/B":      "CDKN2A_B",
    "wildtype":      None,           # reference: all-zeros row, no column
}

# Fixed column order (drives the design matrix; wildtype excluded)
ALTERATION_COLUMNS = ["KIAA1549_BRAF", "BRAF_V600E", "NF1", "FGFR",
                      "RTK", "IDH", "MYB", "other_MAPK", "CDKN2A_B"]

HISTOLOGY_PREFIXES = {"LGG", "GNG", "GNT"}  # dropped; not molecular alterations

def multihot_row(subtype):
    row = {c: 0 for c in ALTERATION_COLUMNS}
    unmapped = []
    for tok in str(subtype).split(","):
        tok = tok.strip()
        if not tok or tok in HISTOLOGY_PREFIXES:
            continue
        if tok not in TOKEN_MAP:
            unmapped.append(tok)
            continue
        col = TOKEN_MAP[tok]
        if col is not None:
            row[col] = 1
    if unmapped:
        print(f"  [WARN] unmapped token(s) in {subtype!r}: {unmapped}")
    return pd.Series(row)

def encode(df):
    mh = df["Molecular Subtype"].apply(multihot_row)
    return pd.concat([df, mh], axis=1)

discovery_molecular = encode(discovery_molecular)
replicate_molecular = encode(replicate_molecular)

# ── Verification ─────────────────────────────────────────────────────────
print("\nAlteration prevalence (Discovery):")
print(discovery_molecular[ALTERATION_COLUMNS].sum().sort_values(ascending=False))
print("\nAlteration prevalence (Replicate):")
print(replicate_molecular[ALTERATION_COLUMNS].sum().sort_values(ascending=False))

n_alt = discovery_molecular[ALTERATION_COLUMNS].sum(axis=1)
print(f"\nDiscovery wildtype (all-zeros) rows: {(n_alt == 0).sum()}")
print("Alterations-per-subject distribution (Discovery):")
print(n_alt.value_counts().sort_index())

# audit table: original string -> which columns fire
audit = X_molecular[["Molecular Subtype"]].copy()
audit = audit.join(pd.concat([
    discovery_molecular[ALTERATION_COLUMNS],
    replicate_molecular[ALTERATION_COLUMNS],
]))
audit.drop_duplicates("Molecular Subtype").to_csv("./molecular_multihot_audit.csv", index=False)

discovery_molecular = discovery_molecular.drop(columns=["Molecular Subtype"])
replicate_molecular = replicate_molecular.drop(columns=["Molecular Subtype"])

X_molecular = pd.concat([discovery_molecular, replicate_molecular])
X_molecular.to_pickle("X_molecular.pkl")
print(f"\nFinal shape: {X_molecular.shape}")
print(f"Alteration features: {ALTERATION_COLUMNS}")

In [ ]:
X_molecular

# 2. Output Features

In [ ]:
# select columns
y = molecular_features[["Progression Free Survival", "Event"]].merge(
    X_molecular["Cohort"].to_frame(), left_index=True, right_index=True
)
# compute age in months
y["Progression Free Survival"] = y["Progression Free Survival"].apply(
    lambda x: int(x) / 30.417
)
# cache output features
y.to_pickle("./y.pkl")
y.shape

In [ ]:
y